# Green MGGP — MGGP bi-objetivo com custo aritmético ponderado pela plataforma

Este notebook usa o pacote **`mggp`** do Rafael ([RafaelGAT108/mggp_model](https://github.com/RafaelGAT108/mggp_model)), sem alterá-lo,
e troca apenas o segundo objetivo pelo **custo aritmético do polinômio executado**, em equivalentes de soma:

$$C_\rho(\mathcal M) = a + \rho\, b,\qquad a=p-1\ \text{somas},\quad b=\sum_{i=1}^{p-1} d_i\ \text{multiplicações},$$

onde $\rho$ é o custo de uma multiplicação relativo a uma soma na plataforma-alvo. Dois cenários (variável `GMGGP_COST`):

| Cenário | $\rho$ | Significado | Saídas |
|---|---|---|---|
| `ops` (**principal**) | 1 | contagem de operações; calibrado pela medição de tempo ($t_\times/t_+\approx1$, ver `Notes/06`) | `Resultados/ops/`, figuras/tabelas sem sufixo, `results_macros.tex` |
| `karatsuba` | $B^{\alpha-1}=729/64\approx11{,}39$ | Green-Box em nível de bits ($B=64$, Karatsuba); $= (Ba+B^\alpha b)/B$ | `Resultados/karatsuba/`, sufixo `_K`, `results_macros_K.tex` (macros com prefixo `Kara`) |

Configurações comparadas nos **exemplos de regressão que já vêm no pacote** (Tutorial 1 e o simulador Bouc–Wen de `utils/utils.py`):

| Nome | O que é |
|---|---|
| `SO` | MGGP mono-objetivo do pacote (só erro; o custo não é usado na busca). |
| `MO-lex` | Modo bi-objetivo do pacote (`new_evaluation` = custo): torneio e Hall of Fame comparam (erro, custo) **lexicograficamente**; a frente de Pareto é extraída só no final. |
| `Green` | **Green MGGP** (proposta): mesmo custo, mas seleção de Pareto (torneio por rank/crowding + sobrevivência NSGA-II elitista, sem estruturas duplicadas). Ver `green_mggp.py`. |

Ordem de execução: `GMGGP_COST=karatsuba` primeiro, depois `GMGGP_COST=ops` (que monta também a comparação entre cenários).
Kernel: **Python (greenmggp)** (`~/.venvs/greenmggp`).


In [ ]:
import os, sys, io, json, time, random, tempfile, contextlib, warnings
warnings.filterwarnings("ignore")
os.environ["TQDM_DISABLE"] = "1"          # silencia as barras de progresso do pacote

import numpy as np
import pandas as pd
import matplotlib as mpl
import matplotlib.pyplot as plt
from scipy.stats import mannwhitneyu

ROOT = os.path.abspath(".")
sys.path.insert(0, ROOT)
sys.path.insert(0, os.path.join(ROOT, "mggp_model"))   # para utils/utils.py (Bouc-Wen)

import mggp
from mggp import MGGP
from green_mggp import GreenMGGP, make_cost, model_size, canonical_terms, COST_SCENARIOS, B_BITS, ALPHA
from utils.utils import simulate                        # simulador Bouc-Wen do pacote

COST = os.environ.get("GMGGP_COST", "ops")            # "ops" (rho = 1, principal) ou "karatsuba" (rho = 729/64)
RHO = COST_SCENARIOS[COST]
COST_FN = make_cost(RHO)
PRIMARY = COST == "ops"
SUF = "" if PRIMARY else "_K"                              # sufixo de figuras/tabelas/rótulos do cenário secundário
MPRE = "" if PRIMARY else "Kara"                           # prefixo das macros do cenário secundário
RES = os.path.join(ROOT, "Resultados", COST)
DRAFT = os.path.join(ROOT, "27Collab-Draft")
FIG = os.path.join(DRAFT, "figures")
TAB = os.path.join(DRAFT, "tables")
for d in (RES, FIG, TAB):
    os.makedirs(d, exist_ok=True)

RERUN = bool(int(os.environ.get("GMGGP_RERUN", "0")))   # True: refaz tudo; False: usa Resultados/runs.json se existir
N_SEEDS = int(os.environ.get("GMGGP_SEEDS", "30"))

# estilo para artigo IEEE (coluna = 3.5 in)
mpl.rcParams.update({
    "font.family": "serif", "font.serif": ["Times New Roman", "Times", "STIXGeneral", "DejaVu Serif"],
    "mathtext.fontset": "stix", "font.size": 8, "axes.labelsize": 8, "axes.titlesize": 8,
    "legend.fontsize": 7, "xtick.labelsize": 7, "ytick.labelsize": 7,
    "axes.linewidth": 0.6, "lines.linewidth": 1.0, "grid.linewidth": 0.4,
    "pdf.fonttype": 42, "ps.fonttype": 42, "savefig.bbox": "tight", "savefig.pad_inches": 0.02,
})
COL = {"SO": "#555555", "MO-lex": "#E69F00", "Green": "#009E73"}
CONFIGS = ["SO", "MO-lex", "Green"]
import re

def save_fig(fig, name):
    fig.savefig(os.path.join(FIG, f"{name}{SUF}.pdf"))

def save_tex(name, lines):
    text = "\n".join(lines) + "\n"
    if SUF:
        text = re.sub(r"\\label\{(tab:[^}]*)\}", lambda m: "\\label{" + m.group(1) + SUF + "}", text)
    open(os.path.join(TAB, f"{name}{SUF}.tex"), "w").write(text)
    return text

print("mggp:", mggp.__file__, "| custo:", COST, "rho =", RHO)

## 1. A função de custo

Cada gene do MGGP é um monômio $\varphi_i=\prod_{j=1}^{d_i} x_{ij}$ (fatores $y$ ou $u$ atrasados pelos operadores $q^{-j}$).
O modelo executado é $\hat y=\theta_0+\sum_i\theta_i\varphi_i$: $a=p-1$ somas e $b=\sum_i d_i$ multiplicações (coeficiente × $d_i$ fatores).
Genes repetidos são fundidos antes de contar (eles colapsam num único coeficiente). Custo: $C_\rho=a+\rho b$.


In [ ]:
from deap import gp
_cwd = os.getcwd(); os.chdir(tempfile.mkdtemp())          # o pacote cria models_saved/ no cwd
pset = MGGP(inputs=np.zeros((50, 1)), outputs=np.zeros((50, 1)), nDelays=2, operators=["mul"]).element._pset
os.chdir(_cwd)
ex = [["y1", "u1", "mul(u1, y1)"], ["y1", "y1", "u1"], ["mul(q1(y1), u1)", "mul(y1, q1(u1))"]]
for genes in ex:
    ind = [gp.PrimitiveTree.from_string(g, pset) for g in genes]
    n, D = model_size(ind)
    print(f"{genes!s:40s} -> a={n}, b={D}, C_rho={COST_FN(ind):8.3f}")
print(f"rho = {RHO:g}   (Karatsuba: B^(alpha-1) = {B_BITS**(ALPHA-1):.4f})")

## 2. Exemplos do pacote

* **E1 — Tutorial 1 (sem ruído)**: exatamente o sistema do `Tutorial_regression.ipynb`/README,
  $y[k]=0{,}65y[k-1]+0{,}35u[k]+0{,}10u[k]y[k-1]$, 300 amostras, 70 % identificação / 30 % validação.
* **E2 — Tutorial 1 com ruído**: mesmo sistema, com ruído branco gaussiano de medição ($\sigma=0{,}02$) na saída.
* **E3 — Histerese Bouc–Wen**: simulador `utils.utils.simulate` do pacote ($A=0{,}9$, $\beta=\gamma=0{,}008$, $d_p=1{,}6$, RK4 com $h=10^{-3}$ s),
  entrada senoidal de 1 Hz com amplitude crescente (identificação) e senoide de 0,8 Hz (validação), decimada por 20.

In [ ]:
def tutorial_data(noise_std=0.0, noise_seed=2027):
    n = 300
    t = np.linspace(0, 20, n)
    u = (np.sin(t) + 0.25 * np.sin(3 * t)).reshape(-1, 1)
    y = np.zeros((n, 1))
    for k in range(1, n):
        y[k, 0] = 0.65 * y[k - 1, 0] + 0.35 * u[k, 0] + 0.10 * u[k, 0] * y[k - 1, 0]
    y = y + noise_std * np.random.default_rng(noise_seed).standard_normal(y.shape)
    s = int(0.70 * n)
    return dict(u_id=u[:s], y_id=y[:s], u_val=u[s:], y_val=y[s:])

def boucwen_data(D=20, h=1e-3):
    def sim(T, f, amp):
        t = np.arange(0, T, h)
        u = amp(t) * np.sin(2 * np.pi * f * t)
        y = simulate(u, np.gradient(u, h))
        return u[::D].reshape(-1, 1), y[::D].reshape(-1, 1)
    u_id, y_id = sim(6.0, 1.0, lambda t: 20 + 5 * t)       # amplitude 20 -> 50
    u_val, y_val = sim(4.0, 0.8, lambda t: 45 + 0 * t)
    return dict(u_id=u_id, y_id=y_id, u_val=u_val, y_val=y_val)

COMMON = dict(problem_type="regression", mode="NARX", evaluationMode="RMSE",
              evaluationType="MShooting", evaluationTypeTest="FreeRun", k=20,
              generations=50, populationSize=100, mutationRate=0.2, crossoverRate=0.9,
              elitePercentage=10, operators=["mul"])
EXAMPLES = {
    "E1": dict(label="Tutorial (noise-free)", data=tutorial_data(0.0), search=dict(nTerms=5, maxHeight=2, nDelays=2)),
    "E2": dict(label="Tutorial (noisy)", data=tutorial_data(0.02), search=dict(nTerms=5, maxHeight=2, nDelays=2)),
    "E3": dict(label="Bouc--Wen hysteresis", data=boucwen_data(), search=dict(nTerms=8, maxHeight=3, nDelays=3)),
}
for key, ex in EXAMPLES.items():
    d, s = ex["data"], ex["search"]
    # custo de referência (definido a priori): nTerms genes, cada um com grau máximo 2^maxHeight
    ex["C_ref"] = s["nTerms"] * (1 + RHO * 2 ** s["maxHeight"])
    print(key, ex["label"], "N_id =", len(d["y_id"]), "N_val =", len(d["y_val"]), "C_ref =", round(ex["C_ref"], 3))

fig, axs = plt.subplots(1, 3, figsize=(7.16, 1.7))
for ax, key in zip(axs, ["E1", "E2", "E3"]):
    d = EXAMPLES[key]["data"]
    if key == "E3":
        ax.plot(d["u_id"], d["y_id"], color="k", lw=0.7, label="identification")
        ax.plot(d["u_val"], d["y_val"], color=COL["Green"], lw=0.7, ls="--", label="validation")
        ax.set_xlabel("$u$"); ax.set_ylabel("$y$")
    else:
        n1 = len(d["y_id"])
        ax.plot(np.arange(n1), d["y_id"], color="k", lw=0.7, label="identification")
        ax.plot(np.arange(n1, n1 + len(d["y_val"])), d["y_val"], color=COL["Green"], lw=0.7, label="validation")
        ax.set_xlabel("sample $k$"); ax.set_ylabel("$y$")
    ax.set_title(f"{key}: {EXAMPLES[key]['label'].replace('--', '–')}")
    ax.grid(alpha=0.3)
axs[2].legend(loc="lower right", frameon=False)
plt.tight_layout(); fig.savefig(os.path.join(FIG, "fig_data.pdf")); plt.show()

## 3. Execução: 3 exemplos × 3 configurações × 30 sementes

Para cada execução guardamos o conjunto **entregue** pelo método: `SO` → o melhor modelo (Hall of Fame);
`MO-lex` e `Green` → a primeira frente de Pareto (`run()` do pacote), sem estruturas repetidas.
Para cada modelo: erro de identificação $J_E$ (RMSE multiple-shooting, $K=20$), custo $C$,
e NRMSE em **simulação livre** na validação. Também guardamos a frente não dominada da população final (`pop_front`), com $C$ calculado *a posteriori* (permite avaliar o `SO` no mesmo plano).

In [ ]:
def nrmse(yd, yp):
    yd, yp = np.ravel(yd), np.ravel(yp)
    if not np.all(np.isfinite(yp)):
        return np.inf
    return float(np.sqrt(np.mean((yd - yp) ** 2)) / np.std(yd))

def term_str(term):
    # monômio canônico -> string com atrasos reais (y1 -> y(k-1-s), u1 -> u(k-s))
    out = []
    for (var, s), pw in term:
        lag = s + (1 if var.startswith("y") else 0)
        name = f"{var[0]}(k-{lag})" if lag else f"{var[0]}(k)"
        out += [name] * pw
    return "*".join(out)

def describe(model, ex, mg):
    d = ex["data"]
    mg.element.compileModel(model)
    model.theta = list(model.leastSquares(d["y_id"], d["u_id"]))
    try:
        yp, yd = model.predict("FreeRun", d["y_val"], d["u_val"])
        val = nrmse(yd, yp)
    except Exception:
        yp, val = None, np.inf
    n, D = model_size(model)
    terms = sorted(canonical_terms(model))
    return dict(J=float(model.fitness.values[0]) / float(np.std(d["y_id"])),   # NRMSE (MS) de identificação
                C=float(COST_FN(model)), p1=n, sumd=D, val=val,
                terms=[term_str(t) for t in terms],
                theta=[float(np.ravel(th)[0]) for th in model.theta],
                genes=[str(g) for g in model])

def unique_by_structure(models):
    seen, out = set(), []
    for m in models:
        k = canonical_terms(m)
        if k not in seen and np.all(np.isfinite(m.fitness.values)):
            seen.add(k); out.append(m)
    return out

def nondominated(points):
    pts = sorted(points, key=lambda r: (r["C"], r["J"]))
    out, best = [], np.inf
    for r in pts:
        if r["J"] < best:
            out.append(r); best = r["J"]
    return out

import hashlib

def traced(base):
    # subclasse que só OBSERVA a população a cada geração (não consome números aleatórios)
    class Traced(base):
        def _record(self):
            sy = float(np.std(self.outputs))
            ok = [m for m in self._pop if m.fitness.valid and np.isfinite(m.fitness.values[0])]
            uniq = unique_by_structure(ok)
            pts = nondominated([dict(J=float(m.fitness.values[0]) / sy, C=float(COST_FN(m))) for m in uniq])
            sig = hashlib.md5(repr([[str(g) for g in ind] for ind in self._pop]).encode()).hexdigest()
            # "empate perigoso": dois indivíduos com o MESMO erro J (inclusive J = inf) e custos C diferentes
            byJ = {}
            for m in self._pop:
                byJ.setdefault(float(m.fitness.values[0]) if m.fitness.valid else None, set()).add(float(COST_FN(m)))
            tie = any(len(cs) > 1 for jv, cs in byJ.items() if jv is not None)
            n_inf = sum(1 for m in self._pop if m.fitness.valid and not np.isfinite(m.fitness.values[0]))
            self.trace.append(dict(front=[(p["C"], p["J"]) for p in pts], distinct=len(uniq), sig=sig, tie=tie, n_inf=n_inf))
        def step(self, gen_number):
            if not hasattr(self, "trace"):
                self.trace = []
                self._record()                      # população inicial
            super().step(gen_number)
            self._record()
    return Traced

def run_one(key, config, seed):
    ex = EXAMPLES[key]; d = ex["data"]
    cls, extra = {"SO": (traced(MGGP), {}), "MO-lex": (traced(MGGP), {"new_evaluation": COST_FN}),
                  "Green": (traced(GreenMGGP), {"rho": RHO})}[config]
    random.seed(seed); np.random.seed(seed)
    cwd = os.getcwd(); tmp = tempfile.mkdtemp(); os.chdir(tmp)       # o pacote grava models_saved/ no cwd
    try:
        t0 = time.time()
        mg = cls(inputs=d["u_id"], outputs=d["y_id"], validation=(d["u_val"], d["y_val"]),
                 filename=f"{key}_{config}_{seed}", **COMMON, **ex["search"], **extra)
        with contextlib.redirect_stdout(io.StringIO()):
            delivered = mg.run()
        runtime = time.time() - t0
    finally:
        os.chdir(cwd)
    delivered = [describe(m, ex, mg) for m in unique_by_structure(delivered)]
    pop = unique_by_structure(mg._pop)
    pop_desc = [dict(J=float(m.fitness.values[0]) / float(np.std(d["y_id"])), C=float(COST_FN(m))) for m in pop]
    return dict(example=key, config=config, seed=seed, runtime=runtime,
                delivered=delivered, pop_front=nondominated(pop_desc), n_distinct_pop=len(pop),
                trace=mg.trace, front1_sizes=getattr(mg, "front1_sizes", []))

RUNS_FILE = os.path.join(RES, "runs.json")
if RERUN or not os.path.exists(RUNS_FILE):
    runs, t0 = [], time.time()
    for key in EXAMPLES:
        for config in CONFIGS:
            for seed in range(N_SEEDS):
                runs.append(run_one(key, config, seed))
            print(f"{key} {config}: done ({time.time() - t0:.0f} s)")
    json.dump(runs, open(RUNS_FILE, "w"), default=float)
    json.dump(dict(cost=COST, rho=RHO, n_seeds=N_SEEDS, date=time.strftime("%Y-%m-%d %H:%M:%S"),
                   total_s=time.time() - t0), open(os.path.join(RES, "run_info.json"), "w"), indent=2)
else:
    runs = json.load(open(RUNS_FILE))
print(len(runs), "runs")

## 4. Métricas

* **Hipervolume** (HV) no plano normalizado $(\text{NRMSE}, C/C_{\rm ref})$ com ponto de referência $(1,1)$ — maior é melhor.
  `HV_id` usa o erro de identificação (o que o algoritmo otimiza); `HV_val` usa o NRMSE de validação em simulação livre do mesmo conjunto entregue.
* **Seleção de um único modelo** (só dados de identificação, regra fixada a priori):
  $\mathcal M_\tau=\arg\min\{C(\mathcal M): J_E(\mathcal M)\le (1+\tau)J_E^{\min}+\eta\}$, $\tau=0{,}05$, $\eta=10^{-6}$ (NRMSE).
  Para `SO` o modelo selecionado é o próprio modelo entregue.
* **Recuperação de estrutura** (E1, E2): o modelo selecionado **contém** os regressores verdadeiros $\{y(k-1),u(k),u(k)y(k-1)\}$ (`sel_extra` = nº de termos extras).
  Obs.: no pacote atual um indivíduo sem nenhum operador $q^{-j}$ tem `lagMax=0` e a regressão falha, então a estrutura verdadeira *exata* (3 termos: $a=3$, $b=4$) não é avaliável; o mínimo atingível é 4 termos ($a=4$, $b=5$).
* Testes: Mann–Whitney bilateral (`Green` vs cada referência), correção de Holm; tamanho de efeito $A_{12}$ de Vargha–Delaney.

In [ ]:
TAU, ETA = 0.05, 1e-6
TRUE_TERMS = sorted(["y(k-1)", "u(k)", "u(k)*y(k-1)"])

def hv2d(points, ref=(1.0, 1.0)):
    pts = sorted([p for p in points if p[0] < ref[0] and p[1] < ref[1]])
    hv, best = 0.0, ref[1]
    for a, b in pts:                      # a = custo normalizado, b = erro
        if b < best:
            hv += (ref[0] - a) * (best - b); best = b
    return hv

def select_tau(models, tau=None):
    tau = TAU if tau is None else tau
    jmin = min(m["J"] for m in models)
    ok = [m for m in models if m["J"] <= (1 + tau) * jmin + ETA]
    return min(ok, key=lambda m: (m["C"], m["J"]))

rows = []
for r in runs:
    ex = EXAMPLES[r["example"]]; cref = ex["C_ref"]; dl = r["delivered"]
    sel = select_tau(dl)
    rows.append(dict(
        example=r["example"], config=r["config"], seed=r["seed"], runtime=r["runtime"],
        n_delivered=len(dl),
        HV_id=hv2d([(m["C"] / cref, m["J"]) for m in dl]),
        HV_val=hv2d([(m["C"] / cref, m["val"]) for m in dl]),
        HV_pop=hv2d([(m["C"] / cref, m["J"]) for m in r["pop_front"]]),
        sel_J=sel["J"], sel_C=sel["C"], sel_val=sel["val"], sel_p1=sel["p1"], sel_sumd=sel["sumd"],
        sel_terms=" + ".join(sel["terms"]), recovered=set(TRUE_TERMS) <= set(sel["terms"]),
        sel_extra=len(set(sel["terms"]) - set(TRUE_TERMS)),
        best_val=min(m["val"] for m in dl),
        front_has_truth=any(set(TRUE_TERMS) <= set(m["terms"]) for m in dl),
    ))
df = pd.DataFrame(rows)
df.to_csv(os.path.join(RES, "per_run_metrics.csv"), index=False)

def a12(x, y):
    x, y = np.asarray(x), np.asarray(y)
    return float(((x[:, None] > y[None, :]).sum() + 0.5 * (x[:, None] == y[None, :]).sum()) / (len(x) * len(y)))

def holm(pvals):
    order = np.argsort(pvals); m = len(pvals); adj = np.empty(m); run = 0.0
    for i, j in enumerate(order):
        run = max(run, (m - i) * pvals[j]); adj[j] = min(1.0, run)
    return adj

METRICS = ["HV_id", "HV_val", "HV_pop", "sel_C", "sel_val"]
tests = []
for key in EXAMPLES:
    for base in ["SO", "MO-lex"]:
        for met in METRICS:
            g = df[(df.example == key) & (df.config == "Green")][met].replace(np.inf, 1e9)
            b = df[(df.example == key) & (df.config == base)][met].replace(np.inf, 1e9)
            p = mannwhitneyu(g, b, alternative="two-sided").pvalue
            p = 1.0 if not np.isfinite(p) else float(p)
            tests.append(dict(example=key, baseline=base, metric=met, p=p, A12=a12(g, b)))
tests = pd.DataFrame(tests)
tests["p_holm"] = holm(tests["p"].values)
tests.to_csv(os.path.join(RES, "stats_tests.csv"), index=False)

summary = df.groupby(["example", "config"]).agg(
    HV_id=("HV_id", "median"), HV_val=("HV_val", "median"), HV_pop=("HV_pop", "median"),
    sel_C=("sel_C", "median"), sel_val=("sel_val", "median"), sel_p1=("sel_p1", "median"),
    recovered=("recovered", "mean"), front_has_truth=("front_has_truth", "mean"), n_delivered=("n_delivered", "median"), runtime=("runtime", "median"))
summary.to_csv(os.path.join(RES, "summary_medians.csv"))
display(summary.round(4))
display(tests.round(5))

## 4b. Verificação empírica dos resultados teóricos e convergência

* **Teorema (redução lexicográfica)**: com a mesma semente, o `MO-lex` deve reproduzir *exatamente* a sequência de populações do `SO`
  enquanto nenhuma comparação envolver erros iguais com custos diferentes. Comparamos a assinatura (hash) da população geração a geração.
* **Teorema (não deterioração da frente)**: no `Green`, se $|U_1|\le\mu$ em todas as gerações, o hipervolume da frente da população é não decrescente.
  Registramos $|U_1|$ e contamos quedas de HV.

In [ ]:
LEVELS = {k: (2 ** v["search"]["maxHeight"] - 1) * v["search"]["nTerms"] * (v["search"]["nTerms"] + 1) // 2 + v["search"]["nTerms"]
          for k, v in EXAMPLES.items()}           # limite superior do nº de níveis de custo L(G, D)

def trace_hv(r):
    cref = EXAMPLES[r["example"]]["C_ref"]
    return np.array([hv2d([(c / cref, j) for c, j in t["front"]]) for t in r["trace"]])

checks = []
for key in EXAMPLES:
    same, first_div, tie_before = 0, [], []
    for seed in range(N_SEEDS):
        a = next(r for r in runs if r["example"] == key and r["config"] == "SO" and r["seed"] == seed)
        b = next(r for r in runs if r["example"] == key and r["config"] == "MO-lex" and r["seed"] == seed)
        sa, sb = [t["sig"] for t in a["trace"]], [t["sig"] for t in b["trace"]]
        div = next((g for g, (x, y) in enumerate(zip(sa, sb)) if x != y), None)
        if div is None:
            same += 1
        else:
            first_div.append(div)
            # houve empate de J com C diferentes na população usada pelo torneio (div-1)
            # ou na usada pela atualização do Hall of Fame (div)?
            tie_before.append(div >= 1 and bool(a["trace"][div - 1]["tie"] or a["trace"][div]["tie"]))
    row = dict(example=key, lex_identical=same, lex_first_div_median=float(np.median(first_div)) if first_div else np.nan,
               lex_div_explained_by_tie=int(np.sum(tie_before)), lex_diverged=len(first_div),
               L_bound=LEVELS[key], mu=COMMON["populationSize"])
    g = [r for r in runs if r["example"] == key and r["config"] == "Green"]
    row["green_max_U1"] = max(max(r["front1_sizes"]) for r in g)
    for cfg in CONFIGS:
        rs = [r for r in runs if r["example"] == key and r["config"] == cfg]
        drops = [int(np.sum(np.diff(trace_hv(r)) < -1e-12)) for r in rs]
        row[f"hv_drop_runs_{cfg}"] = int(np.sum(np.array(drops) > 0))
    checks.append(row)
checks = pd.DataFrame(checks)
checks.to_csv(os.path.join(RES, "theory_checks.csv"), index=False)
display(checks)

# MO-lex entrega exatamente ND(população final do SO) quando as trajetórias coincidem?
eq = []
for key in EXAMPLES:
    for seed in range(N_SEEDS):
        a = df[(df.example == key) & (df.config == "SO") & (df.seed == seed)].iloc[0]
        b = df[(df.example == key) & (df.config == "MO-lex") & (df.seed == seed)].iloc[0]
        eq.append(abs(a.HV_pop - b.HV_id) < 1e-12)
print("fração de pares (SO, MO-lex) com HV(ND(pop. final SO)) == HV(MO-lex):", np.mean(eq))

# divergências: havia >= 2 indivíduos com J = inf nas populações envolvidas?
inf_ties = {}
for key in EXAMPLES:
    n = 0
    for seed in range(N_SEEDS):
        a_ = next(r for r in runs if r["example"] == key and r["config"] == "SO" and r["seed"] == seed)
        b_ = next(r for r in runs if r["example"] == key and r["config"] == "MO-lex" and r["seed"] == seed)
        sa, sb = [t["sig"] for t in a_["trace"]], [t["sig"] for t in b_["trace"]]
        div = next((g for g, (x, y) in enumerate(zip(sa, sb)) if x != y), None)
        if div is not None and div >= 1:
            n += (a_["trace"][div - 1]["n_inf"] >= 2) or (a_["trace"][div]["n_inf"] >= 2)
    inf_ties[key] = int(n)
checks["div_with_inf"] = checks["example"].map(inf_ties)
checks.to_csv(os.path.join(RES, "theory_checks.csv"), index=False)

# Tabela: verificação empírica dos Teoremas 2 e 3
lines = [r"\begin{table}[t]", r"\centering",
         r"\caption{Empirical verification of Theorems~\ref{thm:lex} and~\ref{thm:elitism} over " + str(N_SEEDS) + r" seeds. "
         r"Lexicographic reduction: seeds in which SO and MO-lex produce identical populations in all generations; "
         r"diverging seeds in which the divergence is preceded by an error tie between individuals of different cost, and in which "
         r"the populations involved contain at least two individuals with $J_E=\infty$; median generation of the first divergence. "
         r"Non-deterioration: largest $|U_t^1|$ over all runs and generations, bound $L(G,h)$ of \eqref{eq:levels} and "
         r"population size $\mu$; runs with at least one decrease of the population-front hypervolume.}",
         r"\label{tab:checks}", r"\small", r"\setlength{\tabcolsep}{4pt}",
         r"\begin{tabular}{lcccccccccc}", r"\toprule",
         r" & \multicolumn{4}{c}{Lexicographic reduction} & \multicolumn{3}{c}{Capacity} & \multicolumn{3}{c}{Runs with HV decrease} \\",
         r"\cmidrule(lr){2-5}\cmidrule(lr){6-8}\cmidrule(lr){9-11}",
         r"Ex. & identical & tie & $J_E{=}\infty$ & first div. & $\max|U_t^1|$ & $L(G,h)$ & $\mu$ & SO & MO-lex & Green \\", r"\midrule"]
for _, c in checks.iterrows():
    fd = "--" if not np.isfinite(c["lex_first_div_median"]) else f"{c['lex_first_div_median']:.0f}"
    lines.append(f"{c['example']} & {int(c['lex_identical'])}/{N_SEEDS} & {int(c['lex_div_explained_by_tie'])}/{int(c['lex_diverged'])} & "
                 f"{int(c['div_with_inf'])}/{int(c['lex_diverged'])} & {fd} & {int(c['green_max_U1'])} & {int(c['L_bound'])} & {int(c['mu'])} & "
                 f"{int(c['hv_drop_runs_SO'])} & {int(c['hv_drop_runs_MO-lex'])} & {int(c['hv_drop_runs_Green'])} \\\\")
lines += [r"\bottomrule", r"\end{tabular}", r"\end{table}"]
print(save_tex("table_checks", lines))

In [ ]:
# Fig. convergência: HV da frente da população (com custo calculado para SO a posteriori) e nº de estruturas distintas
fig, axs = plt.subplots(2, 3, figsize=(7.16, 3.0), sharex=True)
for j, key in enumerate(EXAMPLES):
    for cfg in CONFIGS:
        rs = [r for r in runs if r["example"] == key and r["config"] == cfg]
        H = np.array([trace_hv(r) for r in rs]); D = np.array([[t["distinct"] for t in r["trace"]] for r in rs])
        gens = np.arange(1, H.shape[1] + 1)
        for i, M in enumerate([H, D]):
            ax = axs[i, j]
            ax.plot(gens, np.median(M, 0), color=COL[cfg], label=cfg)
            ax.fill_between(gens, np.percentile(M, 25, 0), np.percentile(M, 75, 0), color=COL[cfg], alpha=0.2, lw=0)
    axs[0, j].set_title(f"{key}: {EXAMPLES[key]['label'].replace('--', '–')}")
    axs[1, j].set_xlabel("generation")
    for ax in axs[:, j]:
        ax.grid(alpha=0.3)
axs[0, 0].set_ylabel("HV of population front"); axs[1, 0].set_ylabel("distinct structures")
axs[1, 0].legend(frameon=False, loc="lower right")
plt.tight_layout(); save_fig(fig, "fig_convergence"); plt.show()

In [ ]:
# Fig. sensibilidade à tolerância tau da regra de seleção (tau = 0,05 é o valor declarado; isto NÃO o reajusta)
TAUS = [0.0, 0.01, 0.02, 0.05, 0.1, 0.2, 0.5, 1.0]
sens = []
for r in runs:
    best_val = min(m["val"] for m in r["delivered"])
    for t in TAUS:
        s_ = select_tau(r["delivered"], t)
        sens.append(dict(example=r["example"], config=r["config"], seed=r["seed"], tau=t, C=s_["C"], val=s_["val"],
                         rec=set(TRUE_TERMS) <= set(s_["terms"]), oracle_val=best_val))
sens = pd.DataFrame(sens).replace(np.inf, np.nan)
sens.to_csv(os.path.join(RES, "tau_sensitivity.csv"), index=False)

from matplotlib.lines import Line2D
fig, axs = plt.subplots(2, 2, figsize=(7.16, 2.95), sharex=True)
xt = np.arange(len(TAUS))
dodge = {"MO-lex": -0.09, "Green": 0.09}
for j, key in enumerate(["E2", "E3"]):                  # E1: the selected model does not depend on tau
    for i, met in enumerate(["C", "val"]):
        ax = axs[i, j]
        so = sens[(sens.example == key) & (sens.config == "SO")][met].median()
        ax.axhline(so, color=COL["SO"], lw=1.0, ls=(0, (4, 2)), zorder=1)
        for cfg in ["MO-lex", "Green"]:
            g = sens[(sens.example == key) & (sens.config == cfg)].groupby("tau")[met]
            med, lo, hi = g.median().values, g.quantile(.25).values, g.quantile(.75).values
            x = xt + dodge[cfg]
            ax.vlines(x, lo, hi, color=COL[cfg], lw=1.6, alpha=0.45, zorder=2)
            ax.plot(x, med, "-", color=COL[cfg], lw=1.0, zorder=3)
            ax.scatter(x, med, s=16, marker={"MO-lex": "D", "Green": "o"}[cfg], color=COL[cfg],
                       edgecolor="white", lw=0.5, zorder=4)
        ax.axvline(TAUS.index(TAU), color="0.6", lw=0.6, ls="--", zorder=0)
        ax.grid(alpha=0.3, axis="y"); ax.spines[["top", "right"]].set_visible(False)
    orc = sens[(sens.example == key) & (sens.config == "Green") & (sens.tau == 0)]["oracle_val"].median()
    axs[1, j].axhline(orc, color=COL["Green"], ls=":", lw=1.0, zorder=1)
    axs[0, j].set_title(f"{key}: {EXAMPLES[key]['label'].replace('--', '–')}")
    axs[1, j].set_xticks(xt, [f"{t:g}" for t in TAUS]); axs[1, j].set_xlabel(r"tolerance $\tau$")
axs[0, 0].set_ylabel(r"selected cost $C_\tau$"); axs[1, 0].set_ylabel(r"validation NRMSE$_\tau$")
handles = [Line2D([], [], color=COL["SO"], lw=1.0, ls=(0, (4, 2)), label="SO (independent of $\\tau$)"),
           Line2D([], [], color=COL["MO-lex"], marker="D", ms=4, mec="white", label="MO-lex: median, IQR"),
           Line2D([], [], color=COL["Green"], marker="o", ms=4, mec="white", label="Green: median, IQR"),
           Line2D([], [], color=COL["Green"], ls=":", lw=1.0, label="best validation error in the Green sets")]
plt.tight_layout(rect=(0, 0.07, 1, 1), h_pad=0.5)
fig.legend(handles=handles, frameon=False, loc="lower center", ncol=4, bbox_to_anchor=(0.5, -0.005), columnspacing=1.6)
save_fig(fig, "fig_tau"); plt.show()

rec_tau = sens[sens.example == "E2"].groupby(["config", "tau"])["rec"].mean().unstack()
display(rec_tau)

## 5. Tabelas para o artigo (`27Collab-Draft/tables/`)

In [ ]:
def q(x):
    x = np.asarray(x, float); x = x[np.isfinite(x)]
    return np.median(x), np.percentile(x, 25), np.percentile(x, 75), len(x)

def fmt(v, kind):
    if kind == "hv":   return f"{v:.3f}"
    if kind == "cost": return f"{v:,.{0 if float(RHO).is_integer() and float(v).is_integer() else 1}f}".replace(",", "\\,")
    if kind == "err":  return f"{v:.3f}" if v >= 1e-3 else f"{v:.0e}".replace("e-0", "e-")
    return str(v)

def stars(key, base, met):
    p = tests[(tests.example == key) & (tests.baseline == base) & (tests.metric == met)]["p_holm"].iloc[0]
    return "" if p >= 0.05 else ("$^{\\ast}$" if base == "SO" else "$^{\\dagger}$")

# Tabela 1: configuração experimental
lines = [r"\begin{table}[t]", r"\centering", r"\caption{Examples and search settings; $C_\mathrm{ref}$ for $\rho=" + (f"{RHO:g}" if PRIMARY else r"\rho_K") + r"$ (common: population 100, 50 generations, crossover 0.9, initial mutation 0.2; baseline elite retention 10\%, multiple-shooting horizon $K=20$, primitives \texttt{mul} and $q^{-j}$).}",
         r"\label{tab:setup}", r"\setlength{\tabcolsep}{4pt}", r"\begin{tabular}{lcccccc}", r"\toprule",
         r"Example & $N_\mathrm{id}$ & $N_\mathrm{val}$ & genes & height & $j_{\max}$ & $C_\mathrm{ref}$ \\", r"\midrule"]
for key, ex in EXAMPLES.items():
    s = ex["search"]
    lines.append(f"{key} {ex['label']} & {len(ex['data']['y_id'])} & {len(ex['data']['y_val'])} & {s['nTerms']} & {s['maxHeight']} & {s['nDelays']} & {fmt(ex['C_ref'], 'cost')} \\\\")
lines += [r"\bottomrule", r"\end{tabular}", r"\end{table}"]
save_tex("table_setup", lines)

# Tabela 2: resultados (mediana [IQR] em 30 sementes)
lines = [r"\begin{table*}[t]", r"\centering",
         r"\caption{Results over " + str(N_SEEDS) + r" independent runs" + (r" with $\rho=1$" if PRIMARY else r" with $\rho=\rho_K$") + r": median [interquartile range]. HV: hypervolume of the delivered set (identification or free-run validation error); $C_\tau$, NRMSE$_\tau$: cost and free-run validation NRMSE of the model selected by the $\tau$-rule; Rec.$_\tau$, Rec.$_\mathcal{P}$: fraction of runs in which the selected model, or at least one model of the returned set, contains the three true regressors of \eqref{eq:tutorial}. $^{\ast}$/$^{\dagger}$: Green differs from SO/MO-lex (Mann--Whitney, Holm-adjusted $p<0.05$).}",
         r"\label{tab:results}", r"\footnotesize", r"\setlength{\tabcolsep}{2.5pt}", r"\resizebox{\textwidth}{!}{%", r"\begin{tabular}{llcccccc}", r"\toprule",
         r"Ex. & Method & HV$_\mathrm{id}$ & HV$_\mathrm{val}$ & $C_\tau$ & NRMSE$_\tau$ & Rec.$_\tau$ & Rec.$_\mathcal{P}$ \\", r"\midrule"]
for key, ex in EXAMPLES.items():
    for i, cfg in enumerate(CONFIGS):
        sub = df[(df.example == key) & (df.config == cfg)]
        cells = []
        for met, kind in [("HV_id", "hv"), ("HV_val", "hv"), ("sel_C", "cost"), ("sel_val", "err")]:
            m, lo, hi, _ = q(sub[met])
            st = (stars(key, "SO", met) + stars(key, "MO-lex", met)) if cfg == "Green" else ""
            cells.append(f"{fmt(m, kind)} [{fmt(lo, kind)}, {fmt(hi, kind)}]{st}")
        rec = (f"{sub['recovered'].mean():.2f} & {sub['front_has_truth'].mean():.2f}" if key in ("E1", "E2") else "-- & --")
        name = (f"\\multirow{{3}}{{*}}{{{key}}}" if i == 0 else "")
        meth = r"\textbf{Green}" if cfg == "Green" else cfg
        lines.append(f"{name} & {meth} & " + " & ".join(cells) + f" & {rec} \\\\")
    lines.append(r"\midrule" if key != "E3" else r"\bottomrule")
lines += [r"\end{tabular}}", r"\end{table*}"]
print(save_tex("table_results", lines))

In [ ]:
# Tabela 3: modelos selecionados (corrida com HV_id mediano de cada configuração), E2 e E3
def median_run(key, cfg):
    # corrida representativa: mediana do NRMSE de validação do modelo selecionado (mesma regra para todos os métodos)
    sub = df[(df.example == key) & (df.config == cfg)].sort_values(["sel_val", "seed"]).reset_index(drop=True)
    seed = int(sub.loc[len(sub) // 2, "seed"])
    return next(r for r in runs if r["example"] == key and r["config"] == cfg and r["seed"] == seed)

def latex_terms(terms):
    out = []
    for t in terms:
        f = t.split("*")
        from collections import Counter
        c = Counter(f)
        out.append(" ".join(f"{v}^{{{p}}}" if p > 1 else v for v, p in c.items()))
    return ",\\ ".join(out)

lines = [r"\begin{table}[t]", r"\centering",
         r"\caption{Regressors of the model selected by the $\tau$-rule in the representative run (median validation NRMSE of the selected model) and its free-run validation NRMSE.}",
         r"\label{tab:models}", r"\setlength{\tabcolsep}{3pt}", r"\footnotesize",
         r"\begin{tabular}{llp{8.5cm}rr}", r"\toprule", r"Ex. & Method & Regressors & $C$ & NRMSE \\", r"\midrule"]
for key in ["E2", "E3"]:
    for cfg in CONFIGS:
        r = median_run(key, cfg); sel = select_tau(r["delivered"])
        lines.append(f"{key} & {cfg} & ${latex_terms(sel['terms'])}$ & {fmt(sel['C'], 'cost')} & {sel['val']:.3f} \\\\")
    lines.append(r"\midrule" if key == "E2" else r"\bottomrule")
lines += [r"\end{tabular}", r"\end{table}"]
print(save_tex("table_models", lines))

## 6. Figuras (`27Collab-Draft/figures/`, PDF vetorial)

In [ ]:
# Fig. superfícies de atingimento (Fonseca & Fleming): para cada execução r, A_r(c) = menor erro entre os modelos
# entregues com custo <= c; mostramos mediana (50 %) e quartis (25/75 %) sobre as 30 execuções.
def attainment(models_per_run, key_err, grid):
    A = np.full((len(models_per_run), len(grid)), np.inf)
    for i, ms in enumerate(models_per_run):
        for m in ms:
            A[i, grid >= m["C"]] = np.minimum(A[i, grid >= m["C"]], max(m[key_err], 1e-16))
    q = lambda p: np.quantile(A, p, axis=0, method="inverted_cdf")
    return q(0.25), q(0.5), q(0.75)

fig, axs = plt.subplots(2, 3, figsize=(7.16, 3.3), sharex="col")
for j, key in enumerate(EXAMPLES):
    allC = sorted({m["C"] for r in runs if r["example"] == key for m in r["delivered"]})
    grid = np.array(allC + [allC[-1] * 1.05])
    for i, err in enumerate(["J", "val"]):
        ax = axs[i, j]
        for cfg in CONFIGS:
            ms = [r["delivered"] for r in runs if r["example"] == key and r["config"] == cfg]
            lo, me, hi = attainment(ms, err, grid)
            fin = np.isfinite(me)
            ax.step(grid[fin], me[fin], where="post", color=COL[cfg], lw=1.1, label=cfg)
            band = np.isfinite(lo) & np.isfinite(hi)
            ax.fill_between(grid[band], lo[band], hi[band], step="post", color=COL[cfg], alpha=0.18, lw=0)
        ax.set_yscale("log"); ax.grid(alpha=0.3)
    axs[0, j].set_title(f"{key}: {EXAMPLES[key]['label'].replace('--', '–')}")
    axs[1, j].set_xlabel(r"arithmetic cost $C_\rho$")
axs[0, 0].set_ylabel("identification NRMSE"); axs[1, 0].set_ylabel("validation NRMSE")
axs[0, 0].legend(frameon=False, loc="lower left")
plt.tight_layout(); save_fig(fig, "fig_attainment"); plt.show()

In [ ]:
# Fig. hipervolume (identificação e validação)
fig, axs = plt.subplots(2, 3, figsize=(7.16, 2.9), sharex=True)
for j, key in enumerate(EXAMPLES):
    for i, met in enumerate(["HV_id", "HV_val"]):
        ax = axs[i, j]
        data = [df[(df.example == key) & (df.config == c)][met].values for c in CONFIGS]
        bp = ax.boxplot(data, widths=0.55, patch_artist=True, showfliers=True,
                        medianprops=dict(color="k", lw=1), flierprops=dict(ms=2, mec="0.4"),
                        whiskerprops=dict(lw=0.6), capprops=dict(lw=0.6), boxprops=dict(lw=0.6))
        for patch, c in zip(bp["boxes"], CONFIGS):
            patch.set_facecolor(COL[c]); patch.set_alpha(0.55)
        ax.set_xticks([1, 2, 3], CONFIGS); ax.grid(alpha=0.3, axis="y")
        if j == 0:
            ax.set_ylabel("HV (identification)" if met == "HV_id" else "HV (validation)")
        if i == 0:
            ax.set_title(f"{key}: {EXAMPLES[key]['label'].replace('--', '–')}")
plt.tight_layout(); save_fig(fig, "fig_hv"); plt.show()

In [ ]:
# Fig. simulação livre do modelo selecionado (corrida mediana) — E2 (série) e E3 (laço de histerese)
def rebuild_predict(key, cfg):
    ex = EXAMPLES[key]; d = ex["data"]; r = median_run(key, cfg); sel = select_tau(r["delivered"])
    cls = GreenMGGP if cfg == "Green" else MGGP
    cwd = os.getcwd(); os.chdir(tempfile.mkdtemp())
    try:
        mg = cls(inputs=d["u_id"], outputs=d["y_id"], **{**COMMON, "generations": 1, "populationSize": 2}, **ex["search"])
        model = mg.element.buildModelFromList(sel["genes"])
        mg.element.compileModel(model)
        model.theta = model.leastSquares(d["y_id"], d["u_id"])
        yp, yd = model.predict("FreeRun", d["y_val"], d["u_val"])
    finally:
        os.chdir(cwd)
    return np.ravel(yp), np.ravel(yd), sel

fig, axs = plt.subplots(1, 2, figsize=(7.16, 2.0))
ax = axs[0]
for cfg, ls in [("SO", "-"), ("Green", "--")]:
    yp, yd, sel = rebuild_predict("E2", cfg)
    if cfg == "SO":
        ax.plot(yd, color="0.6", lw=1.6, label="measured")
    ax.plot(yp, ls, color=COL[cfg], lw=1.0, label=f"{cfg}: $C$={sel['C']:g}, NRMSE={sel['val']:.3f}")
ax.set_xlabel("validation sample"); ax.set_ylabel("$y$"); ax.set_title("E2: free-run simulation")
ax.legend(frameon=False, fontsize=6.5, loc="lower left"); ax.grid(alpha=0.3)
ax = axs[1]; uval = np.ravel(EXAMPLES["E3"]["data"]["u_val"])
for cfg, ls in [("SO", "-"), ("Green", "--")]:
    yp, yd, sel = rebuild_predict("E3", cfg)
    uu = uval[-len(yd):]
    if cfg == "SO":
        ax.plot(uu, yd, color="0.6", lw=1.6, label="measured")
    ax.plot(uu, yp, ls, color=COL[cfg], lw=1.0, label=f"{cfg}: $C$={sel['C']:g}, NRMSE={sel['val']:.3f}")
ax.set_xlabel("$u$"); ax.set_ylabel("$y$"); ax.set_title("E3: free-run hysteresis loop")
ax.legend(frameon=False, fontsize=6.5, loc="upper left"); ax.grid(alpha=0.3)
plt.tight_layout(); save_fig(fig, "fig_freerun"); plt.show()

## 7. Números para o texto (`27Collab-Draft/results_macros.tex`)

In [ ]:
def med(key, cfg, met):
    x = df[(df.example == key) & (df.config == cfg)][met].replace(np.inf, np.nan).dropna()
    return float(np.median(x))

mac = {"nSeeds": N_SEEDS, "tauSel": TAU}
for key in EXAMPLES:
    K = key
    for cfg, tag in [("SO", "SO"), ("MO-lex", "Lex"), ("Green", "Green")]:
        mac[f"{K}{tag}HVid"] = f"{med(key, cfg, 'HV_id'):.3f}"
        mac[f"{K}{tag}HVval"] = f"{med(key, cfg, 'HV_val'):.3f}"
        mac[f"{K}{tag}Csel"] = fmt(med(key, cfg, 'sel_C'), "cost")
        mac[f"{K}{tag}VALsel"] = f"{med(key, cfg, 'sel_val'):.3f}"
        mac[f"{K}{tag}Rec"] = f"{100 * df[(df.example == key) & (df.config == cfg)]['recovered'].mean():.0f}"
        mac[f"{K}{tag}RecFront"] = f"{100 * df[(df.example == key) & (df.config == cfg)]['front_has_truth'].mean():.0f}"
        mac[f"{K}{tag}Extra"] = f"{med(key, cfg, 'sel_extra'):.0f}"
        mac[f"{K}{tag}Time"] = f"{med(key, cfg, 'runtime'):.1f}"
    # redução de custo do modelo selecionado: mediana das razões por par de sementes não faz sentido (não pareado);
    # reportamos a razão entre medianas, explicitamente nomeada assim no texto
    mac[f"{K}CostRatioSO"] = f"{100 * (1 - med(key, 'Green', 'sel_C') / med(key, 'SO', 'sel_C')):.0f}"
    mac[f"{K}CostRatioLex"] = f"{100 * (1 - med(key, 'Green', 'sel_C') / med(key, 'MO-lex', 'sel_C')):.0f}"
mac["costMulAdd"] = f"{B_BITS ** (ALPHA - 1):.1f}"
mac["rhoK"] = f"{COST_SCENARIOS['karatsuba']:.2f}"
mac["rhoVal"] = f"{RHO:g}" if PRIMARY else f"{RHO:.2f}"
mac["Ctrue"] = fmt(3 + RHO * 4, "cost")          # estrutura verdadeira: a = 3, b = 4
mac["Cfour"] = fmt(4 + RHO * 5, "cost")          # menor modelo exato avaliável: a = 4, b = 5
for key, ex in EXAMPLES.items():
    mac[f"{key}Cref"] = fmt(ex["C_ref"], "cost")
for _, c in checks.iterrows():
    K = c["example"]
    mac[f"{K}LexSame"] = int(c["lex_identical"]); mac[f"{K}GreenMaxU"] = int(c["green_max_U1"])
    mac[f"{K}Lbound"] = int(c["L_bound"])
    for cfg, tag in [("SO", "SO"), ("MO-lex", "Lex"), ("Green", "Green")]:
        mac[f"{K}{tag}HVdrops"] = int(c[f"hv_drop_runs_{cfg}"])
for cfg, tag in [("SO", "SO"), ("MO-lex", "Lex"), ("Green", "Green")]:
    for t in TAUS:
        v = 100 * sens[(sens.example == "E2") & (sens.config == cfg) & (sens.tau == t)]["rec"].mean()
        mac[f"ECRecTau{tag}{str(t).replace('.', 'p')}"] = f"{v:.0f}"
alpha_names = "ABCDEFGHIJ"
MACROS_FILE = os.path.join(DRAFT, f"results_macros{SUF}.tex")
with open(MACROS_FILE, "w") as f:
    f.write(f"% gerado automaticamente por MGGP_multiobjetivo.ipynb (custo {COST}, rho = {RHO:g}) -- não editar à mão\n")
    for k, v in mac.items():
        name = MPRE + "".join(alpha_names[int(c)] if c.isdigit() else c for c in k)   # macros LaTeX não aceitam dígitos
        f.write(f"\\newcommand{{\\{name}}}{{{v}}}\n")
print(open(MACROS_FILE).read())

## 8. Comparação entre cenários de custo ($\rho=1$ vs Karatsuba)

Só roda no cenário principal (`ops`), depois de `Resultados/karatsuba/` existir. Verifica que o `SO` é idêntico nos dois
cenários (o custo não entra na busca dele), compara as métricas e faz a **avaliação cruzada**: o modelo selecionado em
cada cenário é custeado nos dois pesos ($a+b$ e $a+\rho_K b$). Gera `tables/table_rho.tex` e acrescenta macros com
prefixo `Rho` a `results_macros.tex`.


In [ ]:
K_DIR = os.path.join(ROOT, "Resultados", "karatsuba")
if PRIMARY and os.path.exists(os.path.join(K_DIR, "per_run_metrics.csv")):
    rK = COST_SCENARIOS["karatsuba"]
    dfK = pd.read_csv(os.path.join(K_DIR, "per_run_metrics.csv"))
    both = df.merge(dfK, on=["example", "config", "seed"], suffixes=("", "_K"))
    so = both[both.config == "SO"]
    so_same = bool((so.sel_terms == so.sel_terms_K).all())
    print("SO idêntico nos dois cenários (modelo selecionado):", so_same)
    assert so_same, "SO não deveria depender do custo"
    both["ops"] = both.sel_p1 + both.sel_sumd                 # custo do modelo selecionado com rho = 1
    both["ops_K"] = both.sel_p1_K + both.sel_sumd_K           # modelo selecionado no cenário Karatsuba, custeado com rho = 1
    both["kar"] = both.sel_p1 + rK * both.sel_sumd            # e com rho_K
    both["kar_K"] = both.sel_p1_K + rK * both.sel_sumd_K
    both["same_sel"] = both.sel_terms == both.sel_terms_K
    both.to_csv(os.path.join(RES, "rho_comparison.csv"), index=False)

    def m(sub, col): return float(np.median(sub[col].replace(np.inf, np.nan).dropna()))
    rmac, lines = {}, [r"\begin{table}[t]", r"\centering",
        r"\caption{Effect of the multiplication weight $\rho$ over " + str(N_SEEDS) + r" runs (medians). HV: identification "
        r"hypervolume; $\Delta C_\tau$: reduction of the median selected cost of Green MGGP relative to SO, with the cost of the "
        r"same scenario; $a{+}b$ and $C_{\rho_K}$: operation count and Karatsuba-weighted cost of the selected Green model; "
        r"same: fraction of seeds in which Green MGGP selects the same structure under both weights.}",
        r"\label{tab:rho}", r"\setlength{\tabcolsep}{3.5pt}", r"\small",
        r"\begin{tabular}{llccccccc}", r"\toprule",
        r" & & \multicolumn{3}{c}{HV$_\mathrm{id}$} & & \multicolumn{2}{c}{Green $\mathcal M_\tau$} & \\",
        r"\cmidrule(lr){3-5}\cmidrule(lr){7-8}",
        r"Ex. & weight & SO & MO-lex & Green & $\Delta C_\tau$ & $a{+}b$ & $C_{\rho_K}$ & same \\", r"\midrule"]
    for key in EXAMPLES:
        sub = {c: both[(both.example == key) & (both.config == c)] for c in CONFIGS}
        g = sub["Green"]
        for tag, sfx, wname in [("One", "", r"$\rho=1$"), ("Kara", "_K", r"$\rho_K$")]:
            hv = [m(sub[c], "HV_id" + sfx) for c in CONFIGS]
            red = 100 * (1 - m(g, "sel_C" + sfx) / m(sub["SO"], "sel_C" + sfx))
            ops_, kar_ = m(g, "ops" + sfx), m(g, "kar" + sfx)
            same = 100 * g.same_sel.mean()
            lines.append(f"{key if tag == 'One' else ''} & {wname} & " + " & ".join(f"{v:.3f}" for v in hv)
                         + f" & {red:.0f}\\% & {ops_:.0f} & {kar_:.1f} & " + (f"{same:.0f}\\%" if tag == "One" else "") + r" \\")
            K = key
            rmac[f"Rho{K}{tag}GreenHVid"] = f"{hv[2]:.3f}"; rmac[f"Rho{K}{tag}LexHVid"] = f"{hv[1]:.3f}"
            rmac[f"Rho{K}{tag}SOHVid"] = f"{hv[0]:.3f}"; rmac[f"Rho{K}{tag}CostRed"] = f"{red:.0f}"
            rmac[f"Rho{K}{tag}GreenOps"] = f"{ops_:.0f}"; rmac[f"Rho{K}{tag}GreenKar"] = f"{kar_:.1f}"
        rmac[f"Rho{key}Same"] = f"{100 * g.same_sel.mean():.0f}"
        lines.append(r"\midrule" if key != "E3" else r"\bottomrule")
    lines += [r"\end{tabular}", r"\end{table}"]
    print(save_tex("table_rho", lines))
    with open(MACROS_FILE, "a") as f:
        f.write("% comparação entre cenários de custo\n")
        for k, v in rmac.items():
            name = "".join(alpha_names[int(c)] if c.isdigit() else c for c in k)
            f.write(f"\\newcommand{{\\{name}}}{{{v}}}\n")
    display(both.groupby(["example", "config"])[["ops", "ops_K", "kar", "kar_K", "same_sel"]].median())
else:
    print("comparação entre cenários: só no cenário 'ops' e após rodar 'karatsuba'")
